In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.model_selection import learning_curve, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV, SelectFromModel, SelectFpr, VarianceThreshold, mutual_info_classif, chi2
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

In [2]:
dataset = "C:/Users/tamer/Documents/PhD/ML/enhanced_metabolome.xlsx"
df = pd.read_excel(dataset, sheet_name = 'Paul_50%')
df_val = pd.read_excel(dataset, sheet_name = 'Saqib_50%')

#dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_paul.xlsx"
#dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_saqib.xlsx"

#df = pd.read_excel(dataset_total_path)
#df_val = pd.read_excel(dataset_val_path)

print(df.shape)
print(df_val.shape)

(24, 648)
(12, 648)


In [3]:
Class_A = 'LN'
Class_B = 'SN'

df = df[(df["Class"] == Class_A) | (df["Class"] == Class_B)]
df_val = df_val[(df_val["Class"] == Class_A) | (df_val["Class"] == Class_B)]

In [4]:
def encodage(df):
    code = {
    'LP' : 1,
    'SP' : 0,
    'LN' : 3,
    'SN' : 4
}
# Appliquer ce dictionnaire aux colonnes du dataset, avec la fonction map
    for col in df.select_dtypes('object'):
        df[col] = df[col].map(code)

    return df


def preprocessing(df):
    df = encodage(df)

    X = df.drop(['Class'], axis = 1)
    y = df['Class']

    # compter le nombre d'échantillons restants dans le dataset après avoir été inputé
    print(y.value_counts())

    return X, y

In [5]:
X_total, y_total = preprocessing(df)
X_val, y_val = preprocessing(df_val)

Class
3    6
4    6
Name: count, dtype: int64
Class
3    3
4    3
Name: count, dtype: int64


In [6]:
print("var before log2 transormation : " , X_total.var(axis=0).mean())
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
X_total = log2_transformer.fit_transform(X_total)
X_val = log2_transformer.fit_transform(X_val) 
print("var after log2 transormation : " , X_total.var(axis=0).mean())

var before log2 transormation :  6.378347255133588e+17
var after log2 transormation :  6.050716535554722


In [7]:
def evaluation(model):
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)

    print(confusion_matrix(y_val, ypred))
    print(classification_report(y_val, ypred))

In [8]:
LR_L1 = make_pipeline(StandardScaler(), LogisticRegression(penalty='l1', C=10, random_state = 0, solver = 'liblinear'))
evaluation(LR_L1)

[[3 0]
 [3 0]]
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         3
           4       0.00      0.00      0.00         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      0.50      0.33         6



C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Note : 
# - Intégrer la log2transoform dans le pipeline
# - On peut faire varier la variancethreshold limit 

In [8]:
vars_ = X_total.var(axis=0)
q = np.quantile(vars_, 0.99) 
#selector = VarianceThreshold(threshold = q)
selector = SelectKBest(f_classif, k=138)
preprocessor = make_pipeline(selector)

LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', C=10, random_state = 0, solver = 'liblinear'))
evaluation(LR_L1)

[[3 0]
 [1 2]]
              precision    recall  f1-score   support

           3       0.75      1.00      0.86         3
           4       1.00      0.67      0.80         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6



In [ ]:
    vars_ = X_total.var(axis=0)
    q = np.quantile(vars_, i)
    selector = VarianceThreshold(threshold = q)

In [9]:
acc_total = []

for i in np.arange(0.01, 0.99, 0.01):
    vars_ = X_total.var(axis=0)
    q = np.quantile(vars_, i)
    selector = VarianceThreshold(threshold = q)
    #selector = SelectKBest(f_classif, k=i)
    preprocessor = make_pipeline(selector)
    model = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', C=10, random_state = 0, solver = 'liblinear'))
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)
    acc = model.score(X_val, y_val)
    acc_total.append(acc)
print(acc_total)

[0.6666666666666666, 0.5, 0.3333333333333333, 0.6666666666666666, 0.5, 0.5, 0.5, 0.16666666666666666, 0.5, 0.3333333333333333, 0.5, 0.6666666666666666, 0.5, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.8333333333333334, 0.5, 0.5, 0.8333333333333334, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.8333333333333334, 0.6666666666666666, 0.3333333333333333, 0.5, 0.5, 0.8333333333333334, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.8333333333333334, 0.5, 0.5, 0.6666666666666666, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.8333333333333334, 0.8333333333333334, 0.5, 0.5, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334, 0.8333333333333334]


In [40]:
# 100% accuracy
best = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.3333333333333333, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.16666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.6666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0]
best_2 = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.3333333333333333, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.16666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.6666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.6666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.6666666666666666, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.6666666666666666, 0.5, 0.5, 0.5, 0.5, 0.8333333333333334]
print(len(best_2))

399


In [10]:
for i in range(len(acc_total)):
    if acc_total[i] > 0.8:
        print(i, '\n', acc_total[i], '\n')

14 
 1.0 

23 
 0.8333333333333334 

26 
 0.8333333333333334 

38 
 0.8333333333333334 

43 
 0.8333333333333334 

45 
 1.0 

50 
 0.8333333333333334 

54 
 1.0 

65 
 0.8333333333333334 

66 
 0.8333333333333334 

70 
 1.0 

85 
 0.8333333333333334 

86 
 0.8333333333333334 

87 
 0.8333333333333334 

88 
 0.8333333333333334 

89 
 0.8333333333333334 

90 
 0.8333333333333334 

91 
 0.8333333333333334 

92 
 0.8333333333333334 

93 
 0.8333333333333334 

94 
 0.8333333333333334 

95 
 0.8333333333333334 

96 
 0.8333333333333334 

97 
 0.8333333333333334 

